# 03 · Entraînement des LoRA
Une première LoRA simple (rang 8), puis la comparaison des rangs et des étapes.

### Installation (à lancer à chaque nouvelle session Colab)
Exécution > Modifier le type d'exécution > **GPU T4**.

In [ ]:
%cd /content
!git clone https://github.com/oussioumaima7-star/lora-portes-marocaines.git 2>/dev/null || git -C lora-portes-marocaines pull
%cd /content/lora-portes-marocaines
!pip install -q -r requirements.txt

Les LoRA et les images brutes sont perdues à la fin de la session : on les stocke sur Google Drive et on crée des liens symboliques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lora_projet/{loras,raw,processed}
!rm -rf results/loras data/raw data/processed
!ln -s /content/drive/MyDrive/lora_projet/loras results/loras
!ln -s /content/drive/MyDrive/lora_projet/raw data/raw
!ln -s /content/drive/MyDrive/lora_projet/processed data/processed
# sources.csv et captions.csv sont aussi gardés sur Drive (sinon perdus à la déconnexion)
!D=/content/drive/MyDrive/lora_projet; for f in sources.csv captions.csv; do [ -f $D/$f ] || cp data/$f $D/$f 2>/dev/null; rm -f data/$f; ln -s $D/$f data/$f; done

## 1. Première LoRA (≈ 15-25 min sur T4)

In [ ]:
!python -m src.train_lora --config configs/rank8.yaml

In [ ]:
from src.generate import load_pipeline, LoraSwitcher, generate
pipe = load_pipeline(); sw = LoraSwitcher(pipe)
sw.use('results/loras/rank8/final', 1.0)
imgs, _ = generate(pipe, 'a photo of a mrcdoor, blue door, zellige', seed=0, n=4)
imgs[0]

## 2. Comparaison des rangs
Une session Colab par entraînement si nécessaire.

In [ ]:
for r in [4, 16, 32]:
    !python -m src.train_lora --config configs/rank{r}.yaml

## 3. Entraînement long (surapprentissage)
Checkpoints tous les 250 pas.

In [ ]:
!python -m src.train_lora --config configs/steps_long.yaml

In [ ]:
import json, glob
for f in sorted(glob.glob('results/loras/*/metrics_train.json')):
    print(json.load(open(f)))